<a href="https://colab.research.google.com/github/navap3206-debug/hello-world/blob/main/Lab_R5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio de regresión - 5

|                |   |
:----------------|---|
| **Nombre**     |   |
| **Fecha**      |   |
| **Expediente** |   |

## Validación

Hemos estado usando `train_test_split` en nuestros modelos anteriores.

¿Por qué?

Tenemos que evitar sobreajustar las regresiones de forma en que debemos dividir nuestras muestras en train y test para poder notar esto; así podemos comparar la r2 de nuestras predicciones y llegar un veredicto sobre como se comporta nuestro modelo.

Si la muestra es un subset de la población y queremos generalizar sobre la población, ¿no sería mejor utilizar todos los datos al entrenar un modelo?

NO, porque se estaría entrenando posiblemente con un sesgo el cual será muy dificil denotar posteriormente, es importante tener una selección aleatoria de los datos que nos permitan probar la confinza de nuestro modelo.

El propósito de volver a muestrear dentro de nuestro dataset es tener una idea de qué tan buena podría ser la generalización de nuestro modelo. Imagina un dataset ya separado en dos mitades. Utilizas la primera mitad para entrenar el modelo y pruebas en la segunda mitad; la segunda mitad eran datos invisibles para el modelo al momento de entrenar. Esto nos lleva a tres escenario típicos:

1. Si el modelo hace buenas predicciones en la segunda mitad, significa que la primera mitad era "suficiente" para generalizar.
2. Si el modelo no hace buenas predicciones en la segunda mitad, pero sí en la primera mitad, podría ser que había información importante en la segunda mitad que debió haber sido tomada en cuenta al entrenar, o un problema de overfitting.
3. Si el modelo no hace buenas predicciones en la segunda mitad, y tampoco en la primera mitad, se tendrían que revisar los factores y/o el modelo seleccionado.

El caso ideal sería el 1, pero por estadística los errores y varianzas tienen como entrada el número de muestas, por lo que tenemos menos seguridad de nuestros resutados al usar menos muestras. Si vemos que el modelo generaliza bien podemos unir de nuevo el dataset y entrenar sobre el dataset completo.

En el caso 2 está el problema de que no podemos saber qué información es necesaria para el entrenamiento apropiado del modelo; esto nos lleva a pensar que debemos usar el dataset completo para entrenar, pero esto nos lleva al mismo problema de no saber si el modelo puede generalizar.

El problema sólo incrementa si se tienen hiperparámetros en el modelo (e.g. $\lambda$ en regularización).

## Leave-One-Out Cross Validation

Este método de validación es una colección de $n$ `train-test-split`. Teniendo un dataset de $n$ muestras, la lógica es:
1. Saca una muestra del dataset.
2. Entrena tu modelo con las $n-1$ muestras.
3. Evalúa tu modelo en la muestra que quedó fuera con el métrico que más se ajuste a la aplicación.
4. Regresa la muestra al dataset.
5. Repite 1-4 con muestras diferentes hasta haber hecho el procedimiento $n$ veces para $n$ muestras.
6. Calcula la media y desviación estándar de los métricos guardados.

Con los resultados del proceso de validación podemos saber qué tan bueno podría ser el modelo seleccionado con los datos (con/sin transformaciones).

### Ejercicio 1

Utiliza el dataset `Motor Trend Car Road Tests`. Elimina la columna `model` y entrena 32 modelos diferentes utilizando Leave-One-Out Cross Validation con target `mpg`. Utiliza MSE como métrico.

In [22]:
import pandas as pd
import numpy as np
from google.colab import files
from numpy import random
from sklearn.preprocessing import StandardScaler
import statsmodels.api as SM


# ***EDA*** E IMPORTACIÓN

In [4]:
files = files.upload()

Saving Motor Trend Car Road Tests.xlsx to Motor Trend Car Road Tests.xlsx


In [60]:
dt = pd.read_excel('Motor Trend Car Road Tests.xlsx')
dt.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [61]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   model   32 non-null     object 
 1   mpg     32 non-null     float64
 2   cyl     32 non-null     int64  
 3   disp    32 non-null     float64
 4   hp      32 non-null     int64  
 5   drat    32 non-null     float64
 6   wt      32 non-null     float64
 7   qsec    32 non-null     float64
 8   vs      32 non-null     int64  
 9   am      32 non-null     int64  
 10  gear    32 non-null     int64  
 11  carb    32 non-null     int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 3.1+ KB


In [68]:
vec_y = dt['qsec']


In [62]:
num_escalar = ['mpg','cyl','disp','hp','drat','wt','gear','carb']
categorical_cols = ['vs','am']
matrix_esc = StandardScaler().fit_transform(dt[num_escalar])
X_scaled_df = pd.DataFrame(matrix_esc, columns=num_escalar, index=dt.index)
matrix_2 = pd.concat([X_scaled_df, dt[categorical_cols]], axis=1)
matrix_2


,mpg,cyl,disp,hp,drat,wt,gear,carb,vs,am
0,0.153299,-0.106668,-0.579750,-0.543655,0.576594,-0.620167,0.430331,0.746967,0,1
1,0.153299,-0.106668,-0.579750,-0.543655,0.576594,-0.355382,0.430331,0.746967,0,1
2,0.456737,-1.244457,-1.006026,-0.795570,0.481584,-0.931678,0.430331,-1.140108,1,1
3,0.220730,-0.106668,0.223615,-0.543655,-0.981576,-0.002336,-0.946729,-1.140108,1,0
4,-0.234427,1.031121,1.059772,0.419550,-0.848562,0.231297,-0.946729,-0.511083,0,0
5,-0.335572,-0.106668,-0.046906,-0.617748,-1.589643,0.252064,-0.946729,-1.140108,1,0
6,-0.976163,1.031121,1.059772,1.456847,-0.734549,0.366285,-0.946729,0.746967,0,0
7,0.726459,-1.244457,-0.688779,-1.254944,0.177551,-0.028296,0.430331,-0.511083,1,0
8,0.456737,-1.244457,-0.737144,-0.765933,0.614599,-0.069830,0.430331,-0.511083,1,0
9,-0.150138,-0.106668,-0.517448,-0.351014,0.614599,0.231297,0.430331,0.746967,1,0


RETIRAMOS FILA Y ENTRENAMOS


In [76]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression # Ejemplo de modelo
from sklearn.metrics import mean_squared_error


#Crear una lista para guardar los errores de cada iteración
errores_individuales = []
for i in range(32):

    #separamos
    X_test = matrix_2.iloc[[i]]
    y_test = vec_y.iloc[[i]]


    # drop(i) elimina esa fila
    X_train = matrix_2.drop(i)
    y_train = vec_y.drop(i)

    #entrenar
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)

    #predecimos
    prediccion = modelo.predict(X_test)

    error = mean_squared_error(y_test, prediccion)
    errores_individuales.append(error)

#Manipular los resultados finales
errores_individuales = pd.DataFrame(errores_individuales)
errores_individuales.describe()

,0
count,32.000000
mean,1.005414
std,2.120953
min,0.000263
25%,0.062734
50%,0.225567
75%,1.139628
max,11.650366


Interpreta.




EL modelo tiene un desempeño extraño, ya que sus métricas finales están completamente distorsionadas. Lo que ocurre es que el modelo predice bien casi , pero el dato con error 11.65 desvalanció todo; ese dato aumento el promedio y disparó la desviación estándar.



## K-Folds Cross-Validation

El dataset `Motor Trend Car Road Tests` sólo tiene 32 muestras, y utilizar un modelo sencillo de regresión múltiple hace que usar LOOCV sea muy rápido. El dataset `California Housing` tiene $20640$ muestras para $9$ columnas, entonces realizar un ajuste sobre una transformación o sobre el modelo y luego calcular el impacto esperado podría tomar más tiempo.

La solución propuesta es dividir el dataset en *k* folds (partes iguales), ajustar en *k-1* folds y probar en el restante.

### Ejercicio 2
Utiliza el dataset `California Housing` y haz K-folds Cross Validation con 10 folds. Utiliza el MSE como métrico.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print("Dataset Shape:", housing.data.shape, housing.target.shape)
print("Dataset Features:", housing.feature_names)
print("Dataset Target:", housing.target_names)
X = housing.data
y = housing.target

Interpreta.

## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3